### What one row means to me ?
One row represents one content page aggregated over a single month. It summarizes that page's search visibility, click performance, engagement, and content metadata for that month
### which table  ?
fact_content_daily_performance + dim_content
### time window
I am looking at month 2026-03
### what am i ranking 
I am ranking pages by their review opportunity. Pages with high visibility but relatively poor click-through rate and/or engagement receive higher priority for review.(ctr gap later if possible)
### what am i excluding 
I exclude pages with fewer than 500 monthly impressions because CTR calculated from very small numbers is unstable and can fluctuate due to chance, making those pages unreliable for prioritization.


In [2]:
import duckdb
import os, getpass


con = duckdb.connect()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token: ')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# The path to the daily performance table
FACT_DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"



In [3]:
diagnostic_query = """
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_ids
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
"""

print(con.sql(diagnostic_query).df())


   total_rows  unique_content_ids
0      519606              519606


In [4]:
query = f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks)      AS total_clicks,
        AVG(gsc_avg_position) AS average_position
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY content_hash_id
"""
print(con.sql(query).df().head())


            content_hash_id  total_impressions  total_clicks  average_position
0  content_30fc0ffeed8d67e6             2481.0          18.0          9.745353
1  content_aba5eabbcecaa682               36.0           0.0          9.568182
2  content_e648be18615736ab               18.0           0.0         27.166667
3  content_d003c1bfcc5000b2               38.0           0.0         69.306667
4  content_7c6a1d6c1972146e                5.0           0.0          5.800000


In [5]:
describe_query = f"""
    DESCRIBE SELECT * FROM {FACT_DAILY} LIMIT 1
"""
print(con.sql(describe_query).df())


                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [6]:
# 1. The "Before" Query (No filters)
before_query = f"""
    SELECT COUNT(*) AS rows_before
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
"""

# 2. The "After" Query (Filtering for IS TRUE)
after_query = f"""
    SELECT COUNT(*) AS rows_after
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
"""

# Print both so you can compare!
print("--- Row Count Before Filter ---")
print(con.sql(before_query).df())

print("\n--- Row Count After Filter ---")
print(con.sql(after_query).df())


--- Row Count Before Filter ---
   rows_before
0      9841378

--- Row Count After Filter ---
   rows_after
0      364347


In [7]:
client_diagnostic_query = f"""
    SELECT 
        COUNT(DISTINCT client_hash_id) AS clients_with_ga4_in_march
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND ga4_data_available IS TRUE
"""

print(con.sql(client_diagnostic_query).df())


   clients_with_ga4_in_march
0                         41


In [8]:
master_diagnostic = f"""
SELECT
    COUNT(DISTINCT client_hash_id) AS total_clients,
    COUNT(DISTINCT CASE WHEN gsc_data_available IS TRUE THEN client_hash_id END) AS gsc_clients,
    COUNT(DISTINCT CASE WHEN ga4_data_available IS TRUE THEN client_hash_id END) AS ga4_clients,
    COUNT(DISTINCT CASE
        WHEN gsc_data_available IS TRUE
         AND ga4_data_available IS TRUE
        THEN client_hash_id
    END) AS both_clients
FROM {FACT_DAILY}
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
"""

print(con.sql(master_diagnostic).df())


   total_clients  gsc_clients  ga4_clients  both_clients
0             55           47           41            34


In [ ]:
# ── Step 1: Which clients have GA4 / GSC connected? ──────────────────
# This is a CLIENT-LEVEL check, not a row-level check.
# client_has_ga4 is a property of the client, not the day.
# A client either has GA4 connected or it doesn't — period.

client_list_query = f"""
SELECT
    client_hash_id,
    -- These are client-level flags (same value on every row for a given client)
    ANY_VALUE(client_has_ga4) AS has_ga4,
    ANY_VALUE(client_has_gsc) AS has_gsc
FROM {FACT_DAILY}
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
GROUP BY client_hash_id
"""

all_clients = con.sql(client_list_query).df()
print(f"Total clients in March:       {len(all_clients)}")
print(f"Clients with GA4 connected:   {(all_clients['has_ga4'] == True).sum()}")
print(f"Clients with GSC connected:   {(all_clients['has_gsc'] == True).sum()}")
print(f"Clients with BOTH connected:  {((all_clients['has_ga4'] == True) & (all_clients['has_gsc'] == True)).sum()}")
# client-level (ANY_VALUE) is the trustworthy number, row-level was undercounting because of ordinary daily silence


Total clients in March:       55
Clients with GA4 connected:   36
Clients with GSC connected:   55
Clients with BOTH connected:  36


### Design decision — client-level filter, not row-level

The old query filtered `ga4_data_available IS TRUE` **per row** (per page-day). That dropped 96% of rows — but most of those weren't broken data. They were quiet days where a page simply had no visitors. That's the expected shape: most pages, most days, get zero sessions.

**The fix:** filter at the **client level** (`client_has_ga4 IS TRUE`). This keeps all daily rows for GA4-connected clients — including real zeros — and only excludes clients who never had GA4 connected at all.

A page-month with zero total sessions from a GA4-connected client is a **real signal** (high visibility + zero engagement = strongest review opportunity). A zero from a client without GA4 is unmeasurable — a null wearing a zero's clothes.


In [10]:
# ── Step 2: Aggregate monthly totals, filtered at the CLIENT level ────
# JOIN against the client list instead of row-level IS TRUE filters.
# Real zeros from GA4-connected clients are kept (they contribute 0 to SUM).

feature_query = f"""
WITH measurable_clients AS (
    -- Clients where BOTH GA4 and GSC are connected
    SELECT DISTINCT client_hash_id
    FROM {FACT_DAILY}
    WHERE client_has_ga4 IS TRUE
      AND client_has_gsc IS TRUE
)
SELECT
    f.content_hash_id,
    f.client_hash_id,
    SUM(f.gsc_impressions)        AS total_impressions,
    SUM(f.gsc_clicks)             AS total_clicks,
    AVG(f.gsc_avg_position)       AS average_position,
    SUM(f.ga4_sessions)           AS total_sessions,
    SUM(f.ga4_engaged_sessions)   AS total_engaged_sessions,
    SUM(f.ga4_pageviews)          AS total_pageviews
FROM {FACT_DAILY} f
INNER JOIN measurable_clients mc
    ON f.client_hash_id = mc.client_hash_id
WHERE f.report_date >= '2026-03-01'
  AND f.report_date < '2026-04-01'
GROUP BY f.content_hash_id, f.client_hash_id
"""

monthly_features = con.sql(feature_query).df()
print(f"Pages in monthly feature frame: {len(monthly_features):,}")
print(f"Clients represented:            {monthly_features['client_hash_id'].nunique()}")
print()
print(monthly_features.head(10))


Pages in monthly feature frame: 260,737
Clients represented:            43

            content_hash_id           client_hash_id  total_impressions  \
0  content_7a9d90c6bfe5010b  client_625b6439094e23e4                0.0   
1  content_74112b22d00bcc27  client_625b6439094e23e4                0.0   
2  content_55c5e71cfb8cd214  client_625b6439094e23e4                0.0   
3  content_e24f1741c0eb2375  client_625b6439094e23e4                0.0   
4  content_48c4a73e7a13ca9d  client_625b6439094e23e4                0.0   
5  content_9fe87952c38e1067  client_625b6439094e23e4                0.0   
6  content_a9e8aabacdce9b3e  client_625b6439094e23e4                0.0   
7  content_bb06fcc4fd70ecaa  client_625b6439094e23e4                0.0   
8  content_7f0881846b5c6c32  client_625b6439094e23e4                0.0   
9  content_1da45b1da3f15138  client_625b6439094e23e4                0.0   

   total_clicks  average_position  total_sessions  total_engaged_sessions  \
0           0.0      

In [11]:
# ── Step 3: Compare old row-level filter vs new client-level filter ───

old_row_level = f"""
SELECT COUNT(*) AS rows_old_filter
FROM {FACT_DAILY}
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
"""

new_client_level = f"""
WITH measurable_clients AS (
    SELECT DISTINCT client_hash_id
    FROM {FACT_DAILY}
    WHERE client_has_ga4 IS TRUE
      AND client_has_gsc IS TRUE
)
SELECT COUNT(*) AS rows_new_filter
FROM {FACT_DAILY} f
INNER JOIN measurable_clients mc
    ON f.client_hash_id = mc.client_hash_id
WHERE f.report_date >= '2026-03-01'
  AND f.report_date < '2026-04-01'
"""

old_count = con.sql(old_row_level).df().iloc[0, 0]
new_count = con.sql(new_client_level).df().iloc[0, 0]

print(f"Old filter (row-level IS TRUE):      {old_count:>10,} rows")
print(f"New filter (client-level JOIN):       {new_count:>10,} rows")
print(f"Rows recovered:                      {new_count - old_count:>10,}")
print(f"Retention vs unfiltered (9,841,378):  {new_count / 9_841_378 * 100:.1f}%")


Old filter (row-level IS TRUE):         364,347 rows
New filter (client-level JOIN):        7,704,397 rows
Rows recovered:                       7,340,050
Retention vs unfiltered (9,841,378):  78.3%


In [12]:
# ── Investigation 1: Does the CTE's missing date filter explain 36 vs 43? ──
#
# The measurable_clients CTE has NO date filter:
#   SELECT DISTINCT client_hash_id FROM {FACT_DAILY}
#   WHERE client_has_ga4 IS TRUE AND client_has_gsc IS TRUE
#
# But the ANY_VALUE diagnostic query filtered to March only.
# So the CTE might be pulling in clients from OTHER months
# who had both flags TRUE in, say, February — but not in March.

cte_no_date = f"""
SELECT COUNT(DISTINCT client_hash_id) AS clients_no_date_filter
FROM {FACT_DAILY}
WHERE client_has_ga4 IS TRUE
  AND client_has_gsc IS TRUE
"""

cte_with_date = f"""
SELECT COUNT(DISTINCT client_hash_id) AS clients_march_only
FROM {FACT_DAILY}
WHERE client_has_ga4 IS TRUE
  AND client_has_gsc IS TRUE
  AND report_date >= '2026-03-01' AND report_date < '2026-04-01'
"""

print("CTE without date filter (all months):")
print(con.sql(cte_no_date).df())
print()
print("CTE with March date filter:")
print(con.sql(cte_with_date).df())


CTE without date filter (all months):
   clients_no_date_filter
0                      53

CTE with March date filter:
   clients_march_only
0                  43


In [13]:
# ── Investigation 2: Do any clients have MIXED TRUE/FALSE flags? ──────
#
# If client_has_ga4 = TRUE on some rows and FALSE on others for the
# same client in March, then:
#   - DISTINCT (in the CTE): includes the client (at least one row matches)
#   - ANY_VALUE (in the diagnostic): picks arbitrarily, might pick FALSE
#
# That would make the two queries disagree.

mixed_check = f"""
SELECT
    client_hash_id,
    COUNT(DISTINCT client_has_ga4) AS ga4_distinct_values,
    COUNT(DISTINCT client_has_gsc) AS gsc_distinct_values,
    MIN(client_has_ga4::INT)       AS ga4_min,
    MAX(client_has_ga4::INT)       AS ga4_max,
    MIN(client_has_gsc::INT)       AS gsc_min,
    MAX(client_has_gsc::INT)       AS gsc_max
FROM {FACT_DAILY}
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
GROUP BY client_hash_id
HAVING COUNT(DISTINCT client_has_ga4) > 1
    OR COUNT(DISTINCT client_has_gsc) > 1
"""

mixed_df = con.sql(mixed_check).df()
print(f"Clients with mixed flags in March: {len(mixed_df)}")
if len(mixed_df) > 0:
    print()
    print(mixed_df)
else:
    print("No clients have mixed flags -- the mismatch is NOT from mixed values.")


Clients with mixed flags in March: 10

            client_hash_id  ga4_distinct_values  gsc_distinct_values  ga4_min  \
0  client_3f0ce4d44fe94f3d                    2                    1        0   
1  client_a80fca3f171ed1de                    2                    1        0   
2  client_3ffa76342f366962                    2                    1        0   
3  client_e5c2aa26a8598242                    2                    1        0   
4  client_157ffe4d4a595515                    2                    1        0   
5  client_1a730cb2640a1abf                    2                    1        0   
6  client_73cda7b4e4f265ea                    2                    1        0   
7  client_cd12bcfd98942aa1                    2                    1        0   
8  client_fef1a8f436438636                    2                    1        0   
9  client_b77d0d5f08f05e64                    2                    1        0   

   ga4_max  gsc_min  gsc_max  
0        1        1        1  
1      

In [14]:
# ── Investigation 3: Are all-zero rows isolated to a few clients? ─────
#
# The head(10) showed all zeros from client_625b6439094e23e4.
# Is that one low-activity client, or a widespread pattern?

# Pages per client
pages_per_client = monthly_features.groupby('client_hash_id').size().reset_index(name='page_count')
print("Pages per client:")
print(pages_per_client.sort_values('page_count', ascending=False).to_string())
print()

# How many pages have ALL zeros across engagement columns?
zero_engagement = monthly_features[
    (monthly_features['total_sessions'] == 0) &
    (monthly_features['total_engaged_sessions'] == 0) &
    (monthly_features['total_pageviews'] == 0)
]

print(f"Total pages in feature frame:     {len(monthly_features):,}")
print(f"Pages with zero engagement:        {len(zero_engagement):,}")
print(f"Pages with some engagement:        {len(monthly_features) - len(zero_engagement):,}")
print()

# Which clients own the zero-engagement pages?
zeros_by_client = zero_engagement.groupby('client_hash_id').size().reset_index(name='zero_pages')
total_by_client = monthly_features.groupby('client_hash_id').size().reset_index(name='total_pages')
client_summary = total_by_client.merge(zeros_by_client, on='client_hash_id', how='left')
client_summary['zero_pages'] = client_summary['zero_pages'].fillna(0).astype(int)
client_summary['zero_pct'] = (client_summary['zero_pages'] / client_summary['total_pages'] * 100).round(1)
print("Zero-engagement pages by client:")
print(client_summary.sort_values('zero_pages', ascending=False).to_string(index=False))


Pages per client:
             client_hash_id  page_count
16  client_625b6439094e23e4       31887
12  client_3ffa76342f366962       31108
18  client_73cda7b4e4f265ea       29333
8   client_23a62021009f63c4       14621
17  client_65de48885f4ef01b       13781
30  client_ba65e80a1116ae41       13239
41  client_fef1a8f436438636       11223
10  client_3197e6291363b4db       10690
37  client_e547b89c05043229        9308
27  client_a80fca3f171ed1de        7626
4   client_19b89ee4fe3db6da        6871
11  client_3f0ce4d44fe94f3d        6417
7   client_2094c6eb080311d5        6375
39  client_f623b01661d4bfe4        5878
33  client_cd12bcfd98942aa1        5663
26  client_a60a11451483af1c        5457
6   client_20259bd6705d81d4        4806
3   client_157ffe4d4a595515        4789
42  client_ff644d8251367cbb        4522
28  client_b10cb2997d0c7c86        4325
14  client_4a18d1793d92fb84        4311
13  client_400c21c81c8b46ef        3791
38  client_e5c2aa26a8598242        3530
23  client_9958f0a7ae1